<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TangoFlux-EzAudio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/TangoFlux-EzAudio_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# ⚡ TangoFlux + EzAudio-ControlNet — CRPO 44.1 kHz Stereo SFX & Envelope-Cloned Windows Sound Pack Studio

<p align="left">
  <a href="https://github.com/declare-lab/TangoFlux"><img src="https://img.shields.io/badge/GitHub-declare--lab%2FTangoFlux-181717?logo=github" alt="TangoFlux GitHub"></a>
  <a href="https://github.com/haidog-yaqub/EzAudio"><img src="https://img.shields.io/badge/GitHub-haidog--yaqub%2FEzAudio-181717?logo=github" alt="EzAudio GitHub"></a>
  <a href="https://huggingface.co/declare-lab/TangoFlux"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-TangoFlux%20(CRPO)-violet" alt="TangoFlux HF"></a>
  <a href="https://huggingface.co/OpenSound/EzAudio"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-EzAudio%20%2B%20ControlNet-amber" alt="EzAudio HF"></a>
</p>

Welcome to the dual-engine **TangoFlux + EzAudio-ControlNet** Google Colab studio! This notebook pairs two complementary state-of-the-art sound effect architectures in one unified workspace:

- **⚡ Engine 1: `TangoFlux` (`declare-lab/TangoFlux`, 515M FluxTransformer + CRPO)** — Trained with **CLAP-Ranked Preference Optimization (CRPO)** on a Rectified Flow trajectory over 44.1 kHz stereo Oobleck VAE latents. Synthesizes crisp **44.1 kHz stereo** sound effects (`1s`–`30s`) in **~2.5 seconds** with industry-leading prompt adherence.
- **📈 Engine 2: `EzAudio` + `EzAudio-ControlNet (Energy)` (`OpenSound/EzAudio`)** — Uses an energy/loudness contour ControlNet (`EzAudio_ControlNet(model_name='energy')`) so you can upload **any reference `.wav`** (or pick one of our **6 built-in Classic Windows Rhythm Envelope Contours**) and generate a brand-new sound effect that follows the **exact same rhythm, note timing, and attack/decay curve**!
- **🩹 Seam-Free Audio Inpainting (`EzAudio.editing_audio`)** — Replace or repair any time window (`mask_start`, `mask_length`, `boundary`) inside an existing sound effect.
- **🪟 Dual-Engine Windows 10/11 Sound Event Pack Studio** — Build a complete 16-event Windows Sound Scheme using either **TangoFlux 44.1 kHz Stereo** OR **EzAudio-ControlNet Rhythm-Locked Envelopes**, complete with `Install_Scheme.ps1` + `.reg` installer scripts!

In [ ]:
#@title Step 1: Verify GPU & Install TangoFlux + EzAudio Dependencies
import os
import sys
import subprocess
import torch
from tqdm.auto import tqdm
from IPython.display import clear_output

EZAUDIO_DIR = "/content/EzAudio"

def run_cmd(cmd, check=False):
    print(f"⏳ Running: {cmd}")
    return subprocess.run(cmd, shell=True, check=check)

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    else:
        print("⚠️ No CUDA GPU detected — GPU runtime is strongly recommended.")

    run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1", check=False)

    run_cmd(f"{sys.executable} -m pip install -q git+https://github.com/declare-lab/TangoFlux.git", check=False)

    if not os.path.exists(EZAUDIO_DIR):
        run_cmd(f"git clone --depth 1 https://github.com/haidog-yaqub/EzAudio.git {EZAUDIO_DIR}", check=True)

    pip_pkgs = (
        "einops safetensors soundfile huggingface_hub>=0.26.0 "
        "transformers>=4.46.0 diffusers>=0.31.0 accelerate omegaconf "
        "gradio>=5.0.0 matplotlib scipy librosa torchaudio tqdm"
    )
    run_cmd(f"{sys.executable} -m pip install -q -U {pip_pkgs}", check=False)

    if EZAUDIO_DIR not in sys.path:
        sys.path.insert(0, EZAUDIO_DIR)

    clear_output()
    print("✅ Step 1 Complete: TangoFlux & EzAudio-ControlNet dependencies installed!")
except Exception as e:
    print(f"❌ Installation Error: {e}")
    raise

In [ ]:
#@title Step 2: Mount Google Drive & Configure Persistent Model Cache
import os
import sys

WORKSPACE_DIR = "/content/tangoflux_ezaudio_ws"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.ismount("/content/drive")
except Exception as e:
    print(f"ℹ️ Google Drive not mounted: {e}")

if DRIVE_MOUNTED:
    CACHE_ROOT = "/content/drive/MyDrive/AEI_Audio_Cache/tangoflux_ezaudio"
    OUTPUT_DIR = "/content/drive/MyDrive/AEI_Audio_Exports/TangoFlux_EzAudio"
else:
    CACHE_ROOT = os.path.join(WORKSPACE_DIR, "cache")
    OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")

for d in (CACHE_ROOT, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_ROOT, "torch")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print(f"📂 HF_HOME Cache : {os.environ['HF_HOME']}")
print(f"📂 Output Exports: {OUTPUT_DIR}")

In [ ]:
#@title Step 3: Core TangoFlux + EzAudio-ControlNet Engine & Synthetic Envelope Builder
import os
import gc
import sys
import time
import glob
import torch
import torchaudio
import numpy as np
import soundfile as sf

EZAUDIO_DIR = "/content/EzAudio"
if EZAUDIO_DIR not in sys.path:
    sys.path.insert(0, EZAUDIO_DIR)

# ==============================================================================
# Shared Windows 10/11 Sound Scheme & Zero-Latency UI Mastering Engine
# ==============================================================================
import io
import re
import json
import uuid
import base64
import shutil
import zipfile
from datetime import datetime
import scipy.signal
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_SYSTEM_EVENTS = {
    "WindowsLogon": {
        "label": "Windows Logon / Startup",
        "reg_app": ".Default",
        "duration": 3.0,
        "contour": "warm welcoming ascending chord swell with a gentle shimmering reverb tail",
    },
    "WindowsLogoff": {
        "label": "Windows Logoff / Shutdown",
        "reg_app": ".Default",
        "duration": 2.5,
        "contour": "peaceful resolving descending harmonic chord that fades smoothly into silence",
    },
    "WindowsUnlock": {
        "label": "Windows Unlock / Sign-In",
        "reg_app": ".Default",
        "duration": 1.4,
        "contour": "bright welcoming two-note ascending unlock chime with airy resonance",
    },
    "SystemNotification": {
        "label": "System Notification (Toast)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "crisp pleasant two-note notification chime, clear transient attack, short clean tail",
    },
    "WindowsNotifySystemGeneric": {
        "label": "Action Center / Generic Toast Banner",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "modern crystalline two-tone notification ping with soft harmonic decay",
    },
    "SystemAsterisk": {
        "label": "Asterisk / Info Pop-up",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "soft single crystalline ping with subtle harmonic resonance",
    },
    "SystemExclamation": {
        "label": "Exclamation / Warning Alert",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "attention-grabbing mid-range double pulse chime, alert but non-harsh",
    },
    "SystemHand": {
        "label": "Critical Stop / Error",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "low muted percussive double thud with a tense minor interval, error alert",
    },
    "SystemQuestion": {
        "label": "Question / Confirmation Dialog",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "gentle inquisitive rising two-note harmonic inflection chime",
    },
    "Default": {
        "label": "Default Beep / Fallback Alert",
        "reg_app": ".Default",
        "duration": 0.8,
        "contour": "clean short balanced harmonic ping, neutral interface alert",
    },
    "DeviceConnect": {
        "label": "Device Connect (USB Plug-In)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "upbeat two-tone ascending interval (low note to higher fifth), hardware connected",
    },
    "DeviceDisconnect": {
        "label": "Device Disconnect (USB Unplug)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "smooth two-tone descending interval (high note to lower root), hardware unplugged",
    },
    "DeviceFail": {
        "label": "Device Failed to Connect",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "short stuttering muted low buzz double tap, hardware connection failure",
    },
    "EmptyRecycleBin": {
        "label": "Empty Recycle Bin",
        "reg_app": "Explorer",
        "duration": 1.2,
        "contour": "crisp satisfying paper and glass shimmer dissolve whoosh",
    },
    "WindowsUAC": {
        "label": "User Account Control (UAC Prompt)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "authoritative clean two-note security confirmation chime",
    },
    "LowBatteryAlarm": {
        "label": "Low Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "gentle pulsing two-note reminder tone",
    },
    "CriticalBatteryAlarm": {
        "label": "Critical Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.5,
        "contour": "urgent rapid triple pulse warning chime",
    },
    "MailBeep": {
        "label": "New Mail / Message Beep",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "bright airy bell ding notification",
    },
    "MessageNudge": {
        "label": "Message Nudge / Chat Ping",
        "reg_app": ".Default",
        "duration": 0.9,
        "contour": "playful double tactile pop ping notification",
    },
    "PrintComplete": {
        "label": "Print / Task Complete",
        "reg_app": ".Default",
        "duration": 1.3,
        "contour": "satisfying upbeat three-note completion flourish chime",
    },
    "MenuCommand": {
        "label": "Menu Command / Tactile UI Click",
        "reg_app": ".Default",
        "duration": 0.6,
        "contour": "ultra-short crisp tactile UI click pop transient, dry and immediate",
    },
}

SONIC_THEME_PRESETS = {
    "Frutiger Aero Crystal Glass": (
        "crystalline water-drop glass marimba and airy harmonic bell pads, "
        "clean high-tech Frutiger Aero operating system UI sound effect, studio quality"
    ),
    "Cyberpunk Neon HUD": (
        "futuristic cyberpunk holographic HUD telemetry synthesizer, "
        "crisp digital micro-transient with warm sub-bass resonance, sci-fi UI sound"
    ),
    "Cozy Wooden Kalimba & Mallet": (
        "warm organic rosewood marimba, thumb piano kalimba, and soft felt mallet percussion, "
        "intimate close-mic acoustic UI sound effect"
    ),
    "Sci-Fi Spaceship Bridge": (
        "clean starship computer console telemetry bleep, precision FM synthesis harmonic tone, "
        "futuristic spacecraft interface sound effect"
    ),
    "Retro 16-Bit Chiptune Arcade": (
        "crisp nostalgic 16-bit FM synth arcade game UI sound effect, "
        "bright square and triangle wave arpeggio, clean retro console audio"
    ),
    "Warm Lo-Fi Analog Tape Synth": (
        "warm vintage analog poly-synth and Rhodes electric piano chime with gentle tape saturation, "
        "cozy lo-fi interface sound effect"
    ),
    "Minimalist Tactile Micro-Foley": (
        "ultra-clean modern minimalist UI micro-foley, soft mechanical switch click and muted glass tap, "
        "tight dry studio recording"
    ),
    "Dark Fantasy Crystal & Harp": (
        "ethereal celesta, delicate harp harmonic pluck, and mystical resonant crystal chime, "
        "fantasy RPG interface sound effect"
    ),
}

DEFAULT_SELECTED_WIN_EVENTS = [
    "WindowsLogon",
    "WindowsLogoff",
    "WindowsUnlock",
    "SystemNotification",
    "WindowsNotifySystemGeneric",
    "SystemAsterisk",
    "SystemExclamation",
    "SystemHand",
    "DeviceConnect",
    "DeviceDisconnect",
    "EmptyRecycleBin",
    "WindowsUAC",
]


def build_windows_event_prompt(event_key: str, theme_name: str, custom_style: str = "") -> str:
    ev = WINDOWS_SYSTEM_EVENTS.get(event_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
    style_desc = custom_style.strip() if custom_style and custom_style.strip() else SONIC_THEME_PRESETS.get(
        theme_name, SONIC_THEME_PRESETS["Frutiger Aero Crystal Glass"]
    )
    return f"{ev['contour']}, {style_desc}, high fidelity, no background noise"


def apply_phase_safe_stereo_widener(arr: np.ndarray, sr: int, width: float = 0.22) -> np.ndarray:
    """Upgrade mono (N, 1) to phase-coherent stereo (N, 2) or subtly widen stereo above 350 Hz."""
    if arr.ndim == 1:
        arr = arr[:, None]
    n_samples, ch = arr.shape
    if n_samples < 64 or width <= 0.01:
        return np.repeat(arr, 2, axis=1) if ch == 1 else arr

    nyq = 0.5 * float(sr)
    cutoff = min(max(350.0 / nyq, 0.01), 0.85)
    b, a = scipy.signal.butter(2, cutoff, btype="highpass")

    if ch == 1:
        mono = arr[:, 0]
        high = scipy.signal.lfilter(b, a, mono).astype(np.float32)
        delay_samples = max(1, int(0.00038 * sr))  # ~0.38 ms Haas micro-decorrelation
        side = np.zeros_like(high)
        if delay_samples < n_samples:
            side[delay_samples:] = high[:-delay_samples] - high[delay_samples:]
        side *= float(width)
        left = mono + side
        right = mono - side
        return np.stack([left, right], axis=1).astype(np.float32)
    else:
        mid = 0.5 * (arr[:, 0] + arr[:, 1])
        side = 0.5 * (arr[:, 0] - arr[:, 1])
        side_high = scipy.signal.lfilter(b, a, side).astype(np.float32)
        side_boosted = side + float(width) * 0.5 * side_high
        return np.stack([mid + side_boosted, mid - side_boosted], axis=1).astype(np.float32)


def master_ui_sound(
    audio_np: np.ndarray,
    sr: int,
    target_duration_s: float = None,
    trim_silence: bool = True,
    trim_db: float = -45.0,
    fade_in_ms: float = 1.5,
    fade_out_ms: float = 120.0,
    target_dbfs: float = -1.2,
    stereo_widen: bool = True,
    hpf_hz: float = 40.0,
    pitch_semitones: float = 0.0,
) -> np.ndarray:
    """Zero-latency UI sound mastering: snaps to first transient (<1ms pre-roll), 40Hz sub-rumble filter, optional semitone transpose, 1.5ms click-free micro-fade, phase-safe stereo widening, and true-peak normalization."""
    arr = np.asarray(audio_np, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[:, None]  # (samples, 1)
    elif arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
        arr = arr.T  # (samples, channels)

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr - np.mean(arr, axis=0, keepdims=True)

    # Optional sub-rumble high-pass filter (default 40 Hz) to keep UI chimes crisp
    if hpf_hz and float(hpf_hz) > 10.0 and arr.shape[0] > 64:
        nyq = 0.5 * float(sr)
        cutoff = min(max(float(hpf_hz) / nyq, 0.002), 0.45)
        b_hp, a_hp = scipy.signal.butter(2, cutoff, btype="highpass")
        arr = scipy.signal.lfilter(b_hp, a_hp, arr, axis=0).astype(np.float32)

    # Optional semitone pitch shift via high-precision polyphase resampling
    if pitch_semitones and abs(float(pitch_semitones)) >= 0.05 and arr.shape[0] > 64:
        factor = 2.0 ** (float(pitch_semitones) / 12.0)
        up = 1000
        down = max(100, int(round(1000.0 * factor)))
        gcd = np.gcd(up, down)
        arr = scipy.signal.resample_poly(arr, up // gcd, down // gcd, axis=0).astype(np.float32)

    if trim_silence and arr.shape[0] > int(0.04 * sr):
        mono_env = np.max(np.abs(arr), axis=1)
        diff_env = np.abs(np.diff(mono_env, prepend=mono_env[0]))
        peak = float(np.max(mono_env))
        if peak > 1e-6:
            amp_thresh = peak * (10.0 ** (float(trim_db) / 20.0))
            flux_thresh = float(np.max(diff_env)) * 0.08
            active_idx = np.where((mono_env >= amp_thresh) | (diff_env >= flux_thresh))[0]
            if active_idx.size > 0:
                pre_roll = int(0.0008 * sr)   # 0.8 ms zero-latency pre-roll
                post_roll = int(0.160 * sr)   # 160 ms natural reverb tail retention
                start_i = max(0, int(active_idx[0]) - pre_roll)
                end_i = min(arr.shape[0], int(active_idx[-1]) + post_roll)
                if end_i > start_i + int(0.03 * sr):
                    arr = arr[start_i:end_i]

    if target_duration_s is not None and target_duration_s > 0:
        max_samples = int(float(target_duration_s) * sr)
        if arr.shape[0] > max_samples:
            arr = arr[:max_samples]

    n_samples = arr.shape[0]
    if n_samples == 0:
        return np.zeros((int(0.5 * sr), 2), dtype=np.float32)

    if stereo_widen:
        arr = apply_phase_safe_stereo_widener(arr, sr, width=0.22 if arr.shape[1] == 1 else 0.12)

    fi_samples = min(int((float(fade_in_ms) / 1000.0) * sr), max(1, n_samples // 8))
    if fi_samples > 1:
        fi_curve = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, fi_samples, dtype=np.float32)))
        arr[:fi_samples] *= fi_curve[:, None]

    fo_samples = min(int((float(fade_out_ms) / 1000.0) * sr), max(1, n_samples // 2))
    if fo_samples > 1:
        fo_curve = (0.5 * (1.0 + np.cos(np.linspace(0.0, np.pi, fo_samples, dtype=np.float32)))) ** 1.2
        arr[-fo_samples:] *= fo_curve[:, None]

    peak_val = float(np.max(np.abs(arr)))
    if peak_val > 1e-6:
        target_amp = 10.0 ** (float(target_dbfs) / 20.0)
        arr = arr * (target_amp / peak_val)

    arr = np.clip(arr, -0.999, 0.999)
    return arr if arr.shape[1] > 1 else arr[:, 0]


def save_pcm16_wav(path: str, audio_np: np.ndarray, sr: int, subtype: str = "PCM_16") -> str:
    abs_path = os.path.abspath(path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    sf.write(abs_path, audio_np, int(sr), subtype=subtype)
    # Mirror to AEI_TTS_Out if Google Drive is mounted so companion notebooks (InfiniteTalk, SeedVR2, Audio_PostProcessor) auto-discover it
    if "/content/drive/MyDrive/" in abs_path and "/AEI_TTS_Out/" not in abs_path:
        try:
            mirror_dir = "/content/drive/MyDrive/AEI_TTS_Out/SFX_Exports"
            os.makedirs(mirror_dir, exist_ok=True)
            shutil.copy2(abs_path, os.path.join(mirror_dir, os.path.basename(abs_path)))
        except Exception:
            pass
    return abs_path


def load_audio_file_safe(audio_path: str, target_sr: int = None, mono: bool = False):
    """Robust audio loader using soundfile first, falling back to librosa/torchaudio."""
    try:
        data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
        # data is (T, C) -> transpose to (C, T)
        wav = data.T
    except Exception:
        import librosa
        wav, sr = librosa.load(audio_path, sr=None, mono=False)
        if wav.ndim == 1:
            wav = wav[None, :]
    if mono and wav.shape[0] > 1:
        wav = np.mean(wav, axis=0, keepdims=True)
    if target_sr is not None and int(sr) != int(target_sr):
        gcd = np.gcd(int(sr), int(target_sr))
        wav = scipy.signal.resample_poly(wav, int(target_sr) // gcd, int(sr) // gcd, axis=-1).astype(np.float32)
        sr = int(target_sr)
    return wav, int(sr)


def render_waveform_and_spectrogram(wav_path: str, title: str = "Audio Signal & Spectrogram") -> str:
    """Render a dual-panel waveform + spectrogram plot for visual transient & tail inspection."""
    try:
        data, sr = sf.read(wav_path, always_2d=True)
        mono = np.mean(data, axis=1)
        dur = len(mono) / float(sr)
        t = np.linspace(0.0, dur, len(mono))

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 4.2), facecolor="#0f172a")
        for ax in (ax1, ax2):
            ax.set_facecolor("#1e293b")
            ax.tick_params(colors="#cbd5e1", labelsize=8)
            for spine in ax.spines.values():
                spine.set_color("#334155")

        if data.shape[1] > 1:
            ax1.plot(t, data[:, 0], color="#38bdf8", linewidth=0.8, alpha=0.85, label="L")
            ax1.plot(t, data[:, 1], color="#f472b6", linewidth=0.8, alpha=0.75, label="R")
            ax1.legend(loc="upper right", fontsize=7, facecolor="#0f172a", edgecolor="#334155", labelcolor="#f8fafc")
        else:
            ax1.plot(t, mono, color="#38bdf8", linewidth=0.9)
        ax1.set_xlim(0, max(dur, 0.05))
        ax1.set_ylim(-1.05, 1.05)
        ax1.set_ylabel("Amplitude", color="#e2e8f0", fontsize=8)
        ch_label = "Stereo" if data.shape[1] > 1 else "Mono"
        ax1.set_title(f"{title} ({sr} Hz {ch_label}, {dur:.2f}s)", color="#f8fafc", fontsize=10, fontweight="bold")
        ax1.grid(True, color="#334155", alpha=0.4, linestyle="--")

        nfft = min(1024, max(128, 2 ** int(np.log2(max(len(mono) // 8, 128)))))
        ax2.specgram(mono + 1e-10, NFFT=nfft, Fs=sr, noverlap=nfft // 2, cmap="magma")
        ax2.set_xlim(0, max(dur, 0.05))
        ax2.set_xlabel("Time (seconds)", color="#e2e8f0", fontsize=8)
        ax2.set_ylabel("Freq (Hz)", color="#e2e8f0", fontsize=8)

        plt.tight_layout()
        out_png = os.path.join("/tmp", f"spec_{uuid.uuid4().hex[:8]}.png")
        fig.savefig(out_png, dpi=120, facecolor=fig.get_facecolor())
        plt.close(fig)
        return out_png
    except Exception as e:
        print(f"Spectrogram render warning: {e}")
        return None


def build_audition_soundboard_html(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    embed_base64: bool = False,
) -> str:
    """Build a dark-mode HTML5 WebAudio interactive soundboard with 1-click sequential Play-All auditioning."""
    board_uid = f"sb_{uuid.uuid4().hex[:8]}"
    cards_html = []
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"label": ev_key, "reg_app": ".Default", "duration": 1.0})
        if embed_base64 and os.path.exists(wav_info["path"]):
            with open(wav_info["path"], "rb") as bf:
                b64 = base64.b64encode(bf.read()).decode("ascii")
            audio_src = f"data:audio/wav;base64,{b64}"
        else:
            audio_src = f"Media/{wav_info['filename']}"

        cards_html.append(
            f'<div class="aei-sb-card" style="background:#1e293b;border:1px solid #334155;border-radius:10px;padding:10px 12px;display:flex;flex-direction:column;gap:6px;transition:all 0.2s ease;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<strong style="color:#f8fafc;font-size:13px;">🔔 {ev_meta.get("label", ev_key)}</strong>'
            f'<span style="background:#0f172a;color:#38bdf8;font-size:11px;padding:2px 6px;border-radius:4px;font-family:monospace;">{ev_meta.get("reg_app", ".Default")}\\{ev_key}</span>'
            f'</div>'
            f'<audio controls preload="none" src="{audio_src}" style="width:100%;height:32px;"></audio>'
            f'</div>'
        )
    grid_str = "\n".join(cards_html)
    play_all_js = (
        f"const root=document.getElementById('{board_uid}');"
        f"if(!root)return;"
        f"const audios=Array.from(root.querySelectorAll('audio'));"
        f"const cards=Array.from(root.querySelectorAll('.aei-sb-card'));"
        f"if(root._playing){{root._playing=false;audios.forEach(a=>{{a.pause();a.currentTime=0;}});cards.forEach(c=>c.style.borderColor='#334155');this.textContent='▶ Play All Events Sequentially';return;}}"
        f"root._playing=true;this.textContent='⏹ Stop Sequence';"
        f"let idx=0;const btn=this;"
        f"const next=()=>{{if(!root._playing||idx>=audios.length){{root._playing=false;btn.textContent='▶ Play All Events Sequentially';cards.forEach(c=>c.style.borderColor='#334155');return;}}"
        f"cards.forEach((c,i)=>{{c.style.borderColor=(i===idx)?'#38bdf8':'#334155';}});"
        f"const a=audios[idx++];a.currentTime=0;"
        f"a.onended=()=>{{setTimeout(next,280);}};"
        f"a.onerror=()=>{{setTimeout(next,100);}};"
        f"a.play().catch(()=>setTimeout(next,100));}};"
        f"next();"
    )
    return (
        f'<div id="{board_uid}" style="background:#0f172a;color:#f8fafc;padding:16px;border-radius:12px;border:1px solid #334155;font-family:system-ui,-apple-system,sans-serif;">'
        f'<div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;margin-bottom:12px;">'
        f'<div>'
        f'<h3 style="margin:0 0 4px 0;color:#38bdf8;font-size:16px;">🪟 {scheme_name} — Interactive Windows 10/11 Soundboard</h3>'
        f'<p style="margin:0;color:#94a3b8;font-size:12px;">Theme: <b>{theme_name}</b> · Engine: <b>{model_label}</b> · Events: <b>{len(event_wav_map)}</b></p>'
        f'</div>'
        f'<button type="button" onclick="{play_all_js}" style="background:#0284c7;color:#ffffff;border:none;border-radius:8px;padding:8px 14px;font-size:12px;font-weight:600;cursor:pointer;">▶ Play All Events Sequentially</button>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(270px,1fr));gap:10px;">'
        f'{grid_str}'
        f'</div>'
        f'</div>'
    )


LAST_WINPACK_STATE = {
    "pack_dir": None,
    "scheme_name": "AEI_Custom_Scheme",
    "theme_name": "Frutiger Aero Crystal Glass",
    "model_label": "AEI Sound Studio",
    "output_dir": "/content",
    "event_wav_map": {},
}


def package_windows_sound_scheme_zip(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    output_dir: str,
) -> str:
    """Create a complete Windows 10/11 Sound Scheme ZIP with .wav files, Install/Uninstall .ps1, .reg file, Scheme.ini, soundpack_manifest.json, and audition_board.html."""
    safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", (scheme_name or "AEI_Custom_Sounds").strip())[:32].strip("_") or "AEI_Custom"
    display_name = (scheme_name or "AEI Custom Sounds").strip()[:48]
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = os.path.join(output_dir, f"{safe_id}_Windows_SoundPack_{ts}.zip")

    ps1_lines = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Automated Installer: {display_name}",
        f"# Generated by AEI Colab Notebooks ({model_label})",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'Stop'",
        f"$SchemeId = '{safe_id}'",
        f"$SchemeDisplay = '{display_name}'",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"New-Item -ItemType Directory -Force -Path $TargetDir | Out-Null",
        f"$BackupReg = Join-Path $TargetDir 'Backup_Previous_AppEvents.reg'",
        f"if (-not (Test-Path $BackupReg)) {{ reg export 'HKCU\\AppEvents\\Schemes' $BackupReg /y | Out-Null }}",
        f"Copy-Item -Path (Join-Path $PSScriptRoot 'Media\\*.wav') -Destination $TargetDir -Force",
        f"",
        f"$NamesKey = \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\"",
        f"New-Item -Path $NamesKey -Force | Out-Null",
        f"Set-ItemProperty -Path $NamesKey -Name '(Default)' -Value $SchemeDisplay",
        f"",
        f"$EventMap = @{{",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        ps1_lines.append(f"    '{ev_meta['reg_app']}\\{ev_key}' = '{wav_info['filename']}'")
    ps1_lines += [
        f"}}",
        f"$FirstWav = $null",
        f"foreach ($entry in $EventMap.GetEnumerator()) {{",
        f"    $wavPath = Join-Path $TargetDir $entry.Value",
        f"    if (-not $FirstWav -and (Test-Path $wavPath)) {{ $FirstWav = $wavPath }}",
        f"    $schemeEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\$SchemeId\"",
        f"    $currentEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\.Current\"",
        f"    New-Item -Path $schemeEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $schemeEventKey -Name '(Default)' -Value $wavPath",
        f"    New-Item -Path $currentEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $currentEventKey -Name '(Default)' -Value $wavPath",
        f"}}",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value $SchemeId",
        f"if ($FirstWav) {{ try {{ (New-Object System.Media.SoundPlayer $FirstWav).Play() }} catch {{}} }}",
        f"Write-Host \"Successfully installed and activated Windows Sound Scheme: $SchemeDisplay\" -ForegroundColor Green",
        f"Write-Host \"Prior registry state backed up to: $BackupReg\" -ForegroundColor Cyan",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    uninstall_ps1 = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Uninstaller & Default Restore: {display_name}",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'SilentlyContinue'",
        f"$SchemeId = '{safe_id}'",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value '.Default'",
        f"Get-ChildItem 'HKCU:\\AppEvents\\Schemes\\Apps' -Recurse | Where-Object {{ $_.PSChildName -eq $SchemeId }} | Remove-Item -Recurse -Force",
        f"Remove-Item -Path \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\" -Recurse -Force",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"if (Test-Path $TargetDir) {{ Remove-Item -Path $TargetDir -Recurse -Force }}",
        f"Write-Host \"Restored Windows Default sound scheme and removed $SchemeId.\" -ForegroundColor Yellow",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    reg_lines = [
        "Windows Registry Editor Version 5.00",
        "",
        f"; Windows Sound Scheme: {display_name} ({model_label})",
        f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Names\\{safe_id}]",
        f"@=\"{display_name}\"",
        "",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        escaped_path = f"C:\\\\Windows\\\\Media\\\\{safe_id}\\\\{wav_info['filename']}"
        reg_lines += [
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\{safe_id}]",
            f"@=\"{escaped_path}\"",
            "",
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\.Current]",
            f"@=\"{escaped_path}\"",
            "",
        ]

    ini_lines = [
        "[Scheme]",
        f"Name={display_name}",
        f"SchemeId={safe_id}",
        f"Theme={theme_name}",
        f"Generator={model_label}",
        f"CreatedAt={datetime.now().isoformat()}",
        "",
        "[Events]",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ini_lines.append(f"{ev_key}=Media/{wav_info['filename']}")
        ini_lines.append(f"{ev_key}_Prompt={wav_info.get('prompt', '')}")

    manifest_data = {
        "scheme_name": display_name,
        "scheme_id": safe_id,
        "theme_name": theme_name,
        "generator": model_label,
        "created_at": datetime.now().isoformat(),
        "events": {
            k: {
                "filename": f"Media/{v['filename']}",
                "label": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("label", k),
                "reg_app": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("reg_app", ".Default"),
                "prompt": v.get("prompt", ""),
            }
            for k, v in event_wav_map.items()
        },
    }

    standalone_html = (
        "<!DOCTYPE html>\n<html><head><meta charset='utf-8'>"
        f"<title>{display_name} — Windows Soundboard</title></head>"
        "<body style='background:#020617;margin:24px;'>"
        + build_audition_soundboard_html(display_name, event_wav_map, theme_name, model_label, embed_base64=False)
        + "</body></html>"
    )

    readme_txt = (
        f"====================================================================\n"
        f"  {display_name} — Windows 10 / 11 Sound Event Pack\n"
        f"  Generated with: {model_label}\n"
        f"  Theme Preset:   {theme_name}\n"
        f"====================================================================\n\n"
        f"PREVIEW IN BROWSER FIRST:\n"
        f"  Open 'audition_board.html' in any browser to audition all {len(event_wav_map)} events!\n\n"
        f"OPTION 1: 1-Click PowerShell Install (Recommended, No Admin Required)\n"
        f"--------------------------------------------------------------------\n"
        f"1. Extract this entire ZIP archive to any folder.\n"
        f"2. Right-click 'Install_{safe_id}.ps1' -> 'Run with PowerShell'.\n"
        f"3. The script copies the .wav files to %LOCALAPPDATA%\\WindowsSoundSchemes\\{safe_id},\n"
        f"   registers '{display_name}' in the Windows Sound Control Panel, activates it,\n"
        f"   and opens mmsys.cpl so you can preview or customize individual events!\n"
        f"4. To revert at any time, right-click 'Uninstall_{safe_id}.ps1' -> 'Run with PowerShell'.\n\n"
        f"OPTION 2: Manual Registry (.reg) Import\n"
        f"--------------------------------------------------------------------\n"
        f"1. Copy the 'Media' folder contents into 'C:\\Windows\\Media\\{safe_id}\\'.\n"
        f"2. Double-click 'Install_{safe_id}.reg' to register the sound scheme.\n"
    )

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for ev_key, wav_info in event_wav_map.items():
            zf.write(wav_info["path"], arcname=f"Media/{wav_info['filename']}")
        zf.writestr(f"Install_{safe_id}.ps1", "\r\n".join(ps1_lines))
        zf.writestr(f"Uninstall_{safe_id}.ps1", "\r\n".join(uninstall_ps1))
        zf.writestr(f"Install_{safe_id}.reg", "\r\n".join(reg_lines))
        zf.writestr("Scheme.ini", "\r\n".join(ini_lines))
        zf.writestr("soundpack_manifest.json", json.dumps(manifest_data, indent=2))
        zf.writestr("audition_board.html", standalone_html)
        zf.writestr("README_INSTALL.txt", readme_txt.replace("\n", "\r\n"))

    if event_wav_map:
        first_entry = next(iter(event_wav_map.values()))
        if isinstance(first_entry, dict) and "path" in first_entry:
            LAST_WINPACK_STATE["pack_dir"] = os.path.dirname(first_entry["path"])
    LAST_WINPACK_STATE["scheme_name"] = display_name
    LAST_WINPACK_STATE["theme_name"] = theme_name
    LAST_WINPACK_STATE["model_label"] = model_label
    LAST_WINPACK_STATE["output_dir"] = output_dir
    LAST_WINPACK_STATE["event_wav_map"] = dict(event_wav_map)
    return zip_path


def finalize_rerolled_event_in_pack(
    ev_key: str,
    mastered_wav: np.ndarray,
    sr: int,
    ev_prompt: str,
    output_dir: str,
    scheme_name: str = "",
    theme_name: str = "",
    model_label: str = "",
):
    """Save a re-rolled single Windows event .wav into the active pack state, re-bundle the ZIP, and refresh the soundboard."""
    pack_dir = LAST_WINPACK_STATE.get("pack_dir")
    if not pack_dir or not os.path.exists(pack_dir):
        pack_dir = os.path.join(output_dir, f"winpack_reroll_{int(datetime.now().timestamp())}")
        os.makedirs(pack_dir, exist_ok=True)
        LAST_WINPACK_STATE["pack_dir"] = pack_dir

    fname = f"{ev_key}.wav"
    fpath = os.path.join(pack_dir, fname)
    save_pcm16_wav(fpath, mastered_wav, sr)

    ev_map = dict(LAST_WINPACK_STATE.get("event_wav_map") or {})
    ev_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
    LAST_WINPACK_STATE["event_wav_map"] = ev_map

    s_name = (scheme_name or LAST_WINPACK_STATE.get("scheme_name") or "AEI_Custom_Scheme").strip()
    t_name = (theme_name or LAST_WINPACK_STATE.get("theme_name") or "Custom Theme").strip()
    m_lbl = (model_label or LAST_WINPACK_STATE.get("model_label") or "AEI Sound Studio").strip()

    zip_path = package_windows_sound_scheme_zip(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        output_dir=output_dir,
    )
    LAST_WINPACK_STATE["pack_dir"] = pack_dir
    board_html = build_audition_soundboard_html(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        embed_base64=True,
    )
    spec_img = render_waveform_and_spectrogram(fpath, title=f"Re-Rolled Event: {ev_key}")
    status = (
        f"🔄 Re-rolled **`{ev_key}`** ({WINDOWS_SYSTEM_EVENTS.get(ev_key, {}).get('label', ev_key)}) and updated "
        f"**`{os.path.basename(zip_path)}`** ({len(ev_map)} total events in pack)!"
    )
    return zip_path, fpath, spec_img, board_html, status


ENVELOPE_TEMPLATES = {
    "Single Crisp Ping (1.0s)": "single_ping",
    "Ascending Two-Note Chime — USB Connect (1.1s)": "two_note_up",
    "Descending Two-Note Chime — USB Disconnect (1.1s)": "two_note_down",
    "Double Notification Pulse — Toast Alert (1.2s)": "double_pulse",
    "Triple Urgent Alert — Battery / Warning (1.5s)": "triple_alert",
    "Warm Logon Chord Swell (3.0s)": "logon_swell",
}


def synthesize_reference_envelope_wav(pattern_id: str, sr: int = 24000) -> str:
    """Synthesize a clean harmonic reference guide .wav with a precise amplitude envelope for EzAudio-ControlNet."""
    if pattern_id == "single_ping":
        dur = 1.0
        t = np.linspace(0, dur, int(sr * dur), dtype=np.float32)
        env = np.exp(-t * 5.5) * (1.0 - np.exp(-t * 120.0))
        sig = np.sin(2 * np.pi * 587.33 * t) * env
    elif pattern_id == "two_note_up":
        dur = 1.1
        t = np.linspace(0, dur, int(sr * dur), dtype=np.float32)
        sig = np.zeros_like(t)
        m1 = t < 0.35
        t1 = t[m1]
        sig[m1] = np.sin(2 * np.pi * 523.25 * t1) * np.exp(-t1 * 6.0)
        m2 = t >= 0.32
        t2 = t[m2] - 0.32
        sig[m2] += np.sin(2 * np.pi * 783.99 * t2) * np.exp(-t2 * 4.5)
    elif pattern_id == "two_note_down":
        dur = 1.1
        t = np.linspace(0, dur, int(sr * dur), dtype=np.float32)
        sig = np.zeros_like(t)
        m1 = t < 0.35
        t1 = t[m1]
        sig[m1] = np.sin(2 * np.pi * 783.99 * t1) * np.exp(-t1 * 6.0)
        m2 = t >= 0.32
        t2 = t[m2] - 0.32
        sig[m2] += np.sin(2 * np.pi * 523.25 * t2) * np.exp(-t2 * 4.5)
    elif pattern_id == "double_pulse":
        dur = 1.2
        t = np.linspace(0, dur, int(sr * dur), dtype=np.float32)
        sig = np.zeros_like(t)
        for onset, freq in [(0.02, 659.25), (0.24, 880.0)]:
            mask = t >= onset
            dt = t[mask] - onset
            sig[mask] += np.sin(2 * np.pi * freq * dt) * np.exp(-dt * 6.5)
    elif pattern_id == "triple_alert":
        dur = 1.5
        t = np.linspace(0, dur, int(sr * dur), dtype=np.float32)
        sig = np.zeros_like(t)
        for onset in (0.02, 0.28, 0.54):
            mask = t >= onset
            dt = t[mask] - onset
            sig[mask] += np.sin(2 * np.pi * 740.0 * dt) * np.exp(-dt * 8.0)
    else:  # logon_swell
        dur = 3.0
        t = np.linspace(0, dur, int(sr * dur), dtype=np.float32)
        env = (1.0 - np.exp(-t * 3.0)) * np.exp(-np.maximum(0.0, t - 0.6) * 1.4)
        sig = (
            np.sin(2 * np.pi * 261.63 * t)
            + 0.8 * np.sin(2 * np.pi * 329.63 * t)
            + 0.7 * np.sin(2 * np.pi * 392.00 * t)
            + 0.6 * np.sin(2 * np.pi * 493.88 * t)
        ) * env

    sig = sig / (np.max(np.abs(sig)) + 1e-6) * 0.85
    out_wav = os.path.join("/tmp", f"env_{pattern_id}_{uuid.uuid4().hex[:6]}.wav")
    sf.write(out_wav, sig, sr, subtype="PCM_16")
    return out_wav


class TangoFluxEzAudioStudio:
    """Unified manager for TangoFlux (CRPO 44.1 kHz Stereo) and EzAudio / EzAudio-ControlNet."""

    def __init__(self):
        self.tf_model = None
        self.tf_name = None
        self.ez_model = None
        self.ez_name = None
        self.ez_cn = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        # Persist EzAudio checkpoints on Google Drive across sessions
        drive_ckpts = os.path.join(CACHE_ROOT, "ezaudio_ckpts")
        os.makedirs(drive_ckpts, exist_ok=True)
        repo_ckpts = os.path.join(EZAUDIO_DIR, "ckpts")
        if os.path.exists(EZAUDIO_DIR) and not os.path.exists(repo_ckpts):
            try:
                os.symlink(drive_ckpts, repo_ckpts)
            except Exception:
                pass

    def unload_all(self):
        self.tf_model = None
        self.tf_name = None
        self.ez_model = None
        self.ez_name = None
        self.ez_cn = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def ensure_tangoflux(self, variant: str = "declare-lab/TangoFlux") -> str:
        repo = variant.split(" ")[0].strip()
        if self.tf_model is not None and self.tf_name == repo:
            return f"✅ Using cached `{repo}` (44.1 kHz Stereo CRPO)."
        self.ez_model = None
        self.ez_cn = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        from tangoflux import TangoFluxInference
        self.tf_model = TangoFluxInference(name=repo, device=self.device)
        self.tf_name = repo
        return f"✅ Loaded `{repo}` on `{self.device}`."

    def ensure_ezaudio(self, model_name: str = "s3_xl") -> str:
        m = model_name.split(" ")[0].strip()
        if self.ez_model is not None and self.ez_name == m:
            return f"✅ Using cached `EzAudio ({m})`."
        self.tf_model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        orig_cwd = os.getcwd()
        try:
            os.chdir(EZAUDIO_DIR)
            from api.ezaudio import EzAudio
            self.ez_model = EzAudio(model_name=m, device=self.device)
            self.ez_name = m
        finally:
            os.chdir(orig_cwd)
        return f"✅ Loaded `EzAudio ({m})` on `{self.device}`."

    def ensure_ezaudio_controlnet(self) -> str:
        if self.ez_cn is not None:
            return "✅ Using cached `EzAudio-ControlNet (energy)`."
        self.tf_model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        orig_cwd = os.getcwd()
        try:
            os.chdir(EZAUDIO_DIR)
            from api.controlnet import EzAudio_ControlNet
            self.ez_cn = EzAudio_ControlNet(model_name="energy", device=self.device)
        finally:
            os.chdir(orig_cwd)
        return f"✅ Loaded `EzAudio-ControlNet (energy)` on `{self.device}`."

    @torch.inference_mode()
    def generate_tangoflux(
        self,
        variant: str,
        prompt: str,
        duration: float = 4.0,
        steps: int = 50,
        guidance_scale: float = 4.5,
        seed: int = 42,
    ):
        msg = self.ensure_tangoflux(variant)
        actual_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))
        torch.manual_seed(actual_seed)

        import inspect
        sig = inspect.signature(self.tf_model.generate)
        kw = {"steps": int(steps), "duration": float(duration)}
        if "guidance_scale" in sig.parameters:
            kw["guidance_scale"] = float(guidance_scale)
        wav = self.tf_model.generate(prompt.strip(), **kw)
        if isinstance(wav, torch.Tensor):
            arr = wav.detach().cpu().float().numpy()
        else:
            arr = np.asarray(wav, dtype=np.float32)
        if arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
            arr = arr.T
        return arr, 44100, actual_seed, msg

    @torch.inference_mode()
    def generate_ezaudio_controlnet(
        self,
        prompt: str,
        ref_audio_path: str,
        conditioning_scale: float = 1.0,
        guidance_scale: float = 3.5,
        ddim_steps: int = 50,
        seed: int = 42,
    ):
        msg = self.ensure_ezaudio_controlnet()
        actual_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))
        orig_cwd = os.getcwd()
        try:
            os.chdir(EZAUDIO_DIR)
            sr, audio = self.ez_cn.generate_audio(
                prompt.strip(),
                audio_path=ref_audio_path,
                guidance_scale=float(guidance_scale),
                ddim_steps=int(ddim_steps),
                conditioning_scale=float(conditioning_scale),
                random_seed=actual_seed,
            )
        finally:
            os.chdir(orig_cwd)
        arr = np.asarray(audio, dtype=np.float32).squeeze()
        return arr, int(sr), msg

    @torch.inference_mode()
    def inpaint_ezaudio(self, model_name: str, prompt: str, gt_file: str, mask_start: float, mask_length: float, boundary: float = 2.0):
        msg = self.ensure_ezaudio(model_name)
        orig_cwd = os.getcwd()
        try:
            os.chdir(EZAUDIO_DIR)
            sr, audio = self.ez_model.editing_audio(
                prompt.strip(),
                boundary=float(boundary),
                gt_file=gt_file,
                mask_start=float(mask_start),
                mask_length=float(mask_length),
            )
        finally:
            os.chdir(orig_cwd)
        arr = np.asarray(audio, dtype=np.float32).squeeze()
        return arr, int(sr), msg


TFEZ_STUDIO = TangoFluxEzAudioStudio()
print("✅ Step 3 Complete: TangoFlux + EzAudio-ControlNet studio initialized!")

In [ ]:
#@title Step 4: Launch TangoFlux + EzAudio-ControlNet Gradio Studio (4 Tabs)
import os
import time
import gradio as gr
from IPython.display import clear_output

TF_VARIANTS = [
    "declare-lab/TangoFlux (CRPO Preference-Aligned 44.1 kHz Stereo — Recommended)",
    "declare-lab/TangoFlux-base (Base Rectified Flow 44.1 kHz Stereo)",
]


def ui_tf_generate(
    variant,
    prompt,
    duration,
    steps,
    guidance_scale,
    seed,
    num_takes,
    apply_mastering,
    trim_db,
    fade_out_ms,
):
    try:
        if not prompt or not prompt.strip():
            return None, None, None, "❌ Please enter a sound effect prompt."
        t0 = time.time()
        base_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))
        saved = []
        load_msg = ""
        for i in range(int(num_takes)):
            arr, sr, used_seed, load_msg = TFEZ_STUDIO.generate_tangoflux(
                variant=variant,
                prompt=prompt,
                duration=float(duration),
                steps=int(steps),
                guidance_scale=float(guidance_scale),
                seed=base_seed + i,
            )
            if apply_mastering:
                arr = master_ui_sound(
                    arr, sr, target_duration_s=float(duration), trim_silence=True,
                    trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
                )
            p = os.path.join(OUTPUT_DIR, f"tangoflux_44k_{int(time.time())}_s{used_seed}_v{i+1}.wav")
            save_pcm16_wav(p, arr, sr)
            saved.append(p)

        spec = render_waveform_and_spectrogram(saved[0], title="TangoFlux CRPO (44.1 kHz Stereo)")
        take2 = saved[1] if len(saved) > 1 else None
        return saved[0], take2, spec, (
            f"✅ Generated **{len(saved)}** 44.1 kHz stereo clip(s) in **{time.time()-t0:.2f}s** (Seed `{base_seed}`).\n- {load_msg}"
        )
    except Exception as e:
        return None, None, None, f"❌ TangoFlux Error: {e}"


def ui_ez_controlnet(
    envelope_source,
    synthetic_template,
    uploaded_ref,
    prompt,
    cond_scale,
    guidance_scale,
    ddim_steps,
    seed,
    apply_mastering,
    trim_db,
    fade_out_ms,
):
    try:
        if not prompt or not prompt.strip():
            return None, None, None, "❌ Please enter a target sound effect prompt."
        t0 = time.time()
        if "Synthetic" in envelope_source or not uploaded_ref or not os.path.exists(uploaded_ref):
            pat_id = ENVELOPE_TEMPLATES.get(synthetic_template, "two_note_up")
            ref_path = synthesize_reference_envelope_wav(pat_id)
        else:
            ref_path = uploaded_ref

        arr, sr, load_msg = TFEZ_STUDIO.generate_ezaudio_controlnet(
            prompt=prompt,
            ref_audio_path=ref_path,
            conditioning_scale=float(cond_scale),
            guidance_scale=float(guidance_scale),
            ddim_steps=int(ddim_steps),
            seed=int(seed),
        )
        if apply_mastering:
            arr = master_ui_sound(
                arr, sr, trim_silence=True, trim_db=float(trim_db),
                fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
        out_wav = os.path.join(OUTPUT_DIR, f"ezaudio_cn_{int(time.time())}.wav")
        save_pcm16_wav(out_wav, arr, sr)
        spec = render_waveform_and_spectrogram(out_wav, title="EzAudio-ControlNet Envelope-Guided Output")
        return out_wav, ref_path, spec, (
            f"✅ Generated envelope-locked sound effect in **{time.time()-t0:.2f}s**!\n- {load_msg}\n- Output: `{out_wav}`"
        )
    except Exception as e:
        return None, None, None, f"❌ EzAudio-ControlNet Error: {e}"


def ui_ez_inpaint(model_name, src_audio, prompt, mask_start, mask_length, boundary):
    try:
        if not src_audio or not os.path.exists(src_audio):
            return None, None, "❌ Upload a source audio file first."
        t0 = time.time()
        arr, sr, load_msg = TFEZ_STUDIO.inpaint_ezaudio(
            model_name=model_name,
            prompt=prompt or "clean sound effect transition",
            gt_file=src_audio,
            mask_start=float(mask_start),
            mask_length=float(mask_length),
            boundary=float(boundary),
        )
        out_wav = os.path.join(OUTPUT_DIR, f"ezaudio_inpaint_{int(time.time())}.wav")
        save_pcm16_wav(out_wav, arr, sr)
        spec = render_waveform_and_spectrogram(out_wav, title=f"EzAudio Inpainted ({mask_start}s + {mask_length}s)")
        return out_wav, spec, f"✅ Inpainted audio in **{time.time()-t0:.2f}s** (`{out_wav}`)"
    except Exception as e:
        return None, None, f"❌ EzAudio Inpainting Error: {e}"


def ui_tfez_windows_pack(
    engine_mode,
    scheme_name,
    theme_name,
    custom_style,
    selected_events,
    steps,
    guidance_scale,
    base_seed,
    trim_db,
    fade_out_ms,
):
    try:
        if not selected_events:
            return None, None, None, "", "❌ Select at least one Windows system event."
        t0 = time.time()
        pack_dir = os.path.join(OUTPUT_DIR, f"tfez_winpack_{int(time.time())}")
        os.makedirs(pack_dir, exist_ok=True)
        event_wav_map = {}
        first_wav = None
        seed_val = int(base_seed) if int(base_seed) >= 0 else 42
        use_cn = "ControlNet" in engine_mode

        event_to_env = {
            "WindowsLogon": "logon_swell",
            "WindowsLogoff": "two_note_down",
            "DeviceConnect": "two_note_up",
            "DeviceDisconnect": "two_note_down",
            "SystemNotification": "double_pulse",
            "WindowsNotifySystemGeneric": "double_pulse",
            "WindowsUnlock": "two_note_up",
            "CriticalBatteryAlarm": "triple_alert",
            "SystemExclamation": "double_pulse",
            "SystemHand": "double_pulse",
        }

        for idx, ev_key in enumerate(selected_events):
            ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
            dur = float(ev_meta["duration"])
            ev_prompt = build_windows_event_prompt(ev_key, theme_name, custom_style)

            if use_cn:
                pat = event_to_env.get(ev_key, "single_ping")
                ref_env_wav = synthesize_reference_envelope_wav(pat)
                arr, sr, _ = TFEZ_STUDIO.generate_ezaudio_controlnet(
                    prompt=ev_prompt,
                    ref_audio_path=ref_env_wav,
                    conditioning_scale=1.0,
                    guidance_scale=float(guidance_scale),
                    ddim_steps=int(steps),
                    seed=seed_val + idx,
                )
            else:
                arr, sr, _, _ = TFEZ_STUDIO.generate_tangoflux(
                    variant="declare-lab/TangoFlux",
                    prompt=ev_prompt,
                    duration=max(dur + 0.4, 1.5),
                    steps=int(steps),
                    guidance_scale=float(guidance_scale),
                    seed=seed_val + idx,
                )

            mastered = master_ui_sound(
                arr, sr, target_duration_s=dur, trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
            fname = f"{ev_key}.wav"
            fpath = os.path.join(pack_dir, fname)
            save_pcm16_wav(fpath, mastered, sr)
            event_wav_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
            if first_wav is None:
                first_wav = fpath

        zip_path = package_windows_sound_scheme_zip(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=f"{theme_name} ({engine_mode})",
            model_label=engine_mode,
            output_dir=OUTPUT_DIR,
        )
        board_html = build_audition_soundboard_html(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=f"{theme_name} ({engine_mode})",
            model_label=engine_mode,
            embed_base64=True,
        )
        spec_img = render_waveform_and_spectrogram(first_wav, title=f"Windows Event Preview: {selected_events[0]}") if first_wav else None
        return zip_path, first_wav, spec_img, board_html, (
            f"✅ Packaged **{len(event_wav_map)}** Windows events into `{os.path.basename(zip_path)}` in **{time.time()-t0:.1f}s**!"
        )
    except Exception as e:
        return None, None, None, "", f"❌ Windows Sound Pack Error: {e}"


def ui_tfez_reroll_single_event(
    engine_mode,
    scheme_name,
    theme_name,
    custom_style,
    ev_key,
    ev_custom_prompt,
    reroll_seed,
    pitch_semitones,
    steps,
    guidance_scale,
    trim_db,
    fade_out_ms,
):
    try:
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
        dur = float(ev_meta["duration"])
        ev_prompt = (ev_custom_prompt or "").strip() or build_windows_event_prompt(ev_key, theme_name, custom_style)
        use_cn = "ControlNet" in engine_mode
        event_to_env = {
            "WindowsLogon": "logon_swell",
            "WindowsLogoff": "two_note_down",
            "DeviceConnect": "two_note_up",
            "DeviceDisconnect": "two_note_down",
            "SystemNotification": "double_pulse",
            "WindowsNotifySystemGeneric": "double_pulse",
            "WindowsUnlock": "two_note_up",
            "CriticalBatteryAlarm": "triple_alert",
            "SystemExclamation": "double_pulse",
            "SystemHand": "double_pulse",
        }
        if use_cn:
            pat = event_to_env.get(ev_key, "single_ping")
            ref_env_wav = synthesize_reference_envelope_wav(pat)
            arr, sr, _ = TFEZ_STUDIO.generate_ezaudio_controlnet(
                prompt=ev_prompt,
                ref_audio_path=ref_env_wav,
                conditioning_scale=1.0,
                guidance_scale=float(guidance_scale),
                ddim_steps=int(steps),
                seed=int(reroll_seed),
            )
        else:
            arr, sr, _, _ = TFEZ_STUDIO.generate_tangoflux(
                variant="declare-lab/TangoFlux",
                prompt=ev_prompt,
                duration=max(dur + 0.4, 1.5),
                steps=int(steps),
                guidance_scale=float(guidance_scale),
                seed=int(reroll_seed),
            )
        mastered = master_ui_sound(
            arr, sr, target_duration_s=dur, trim_silence=True,
            trim_db=float(trim_db), fade_in_ms=1.5, fade_out_ms=float(fade_out_ms),
            pitch_semitones=float(pitch_semitones),
        )
        return finalize_rerolled_event_in_pack(
            ev_key=ev_key,
            mastered_wav=mastered,
            sr=sr,
            ev_prompt=ev_prompt,
            output_dir=OUTPUT_DIR,
            scheme_name=scheme_name,
            theme_name=f"{theme_name} ({engine_mode})",
            model_label=engine_mode,
        )
    except Exception as e:
        return None, None, None, "", f"❌ Single-Event Re-Roll Error: {e}"


with gr.Blocks(title="TangoFlux + EzAudio-ControlNet SFX Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## ⚡ TangoFlux (CRPO 44.1 kHz Stereo) + EzAudio-ControlNet (Envelope-Guided) SFX Studio\n"
        "Combine **TangoFlux**'s fast preference-aligned 44.1 kHz stereo flow matching with **EzAudio-ControlNet**'s "
        "exact loudness-contour cloning and audio inpainting!"
    )
    banner = gr.Markdown("⏳ Initializing TangoFlux + EzAudio studio...")

    with gr.Tabs():
        with gr.Tab("⚡ 1. TangoFlux CRPO 44.1 kHz Stereo Studio"):
            with gr.Row():
                with gr.Column(scale=3):
                    tf_var = gr.Dropdown(
                        choices=TF_VARIANTS,
                        value=TF_VARIANTS[0],
                        label="🎛️ TangoFlux Checkpoint",
                        info="CRPO checkpoint is preference-aligned for superior prompt faithfulness.",
                    )
                    tf_preset = gr.Dropdown(
                        choices=[
                            "Two-note ascending crystalline glass marimba notification chime with soft reverb",
                            "Crisp wooden kalimba pluck followed by a warm low marimba resonance",
                            "Futuristic sci-fi HUD confirmation beep, clean stereo interface sound",
                            "Heavy metallic vault door unlatching with pneumatic air release",
                            "Hammer slowly hitting a resonant wooden table",
                        ],
                        value="Two-note ascending crystalline glass marimba notification chime with soft reverb",
                        label="✨ Quick SFX Prompt Preset",
                        info="Select a preset to populate the prompt box.",
                    )
                    tf_prompt = gr.Textbox(
                        value="Two-note ascending crystalline glass marimba notification chime with soft reverb",
                        lines=3,
                        label="📝 Sound Effect Prompt",
                        info="TangoFlux excels at multi-part temporal descriptions ('X followed by Y').",
                    )
                    with gr.Row():
                        tf_dur = gr.Slider(1.0, 30.0, value=3.0, step=0.5, label="⏱️ Duration (Seconds, 1s–30s)", info="Target stereo clip length.")
                        tf_steps = gr.Slider(15, 100, value=50, step=5, label="⚡ Rectified Flow Steps", info="25 steps for speed; 50 steps for top stereo clarity.")
                        tf_cfg = gr.Slider(1.0, 9.0, value=4.5, step=0.5, label="🧭 Guidance Scale", info="Default: 4.5.")
                    with gr.Row():
                        tf_seed = gr.Number(value=42, precision=0, label="🎲 Seed (-1 = Random)", info="Seed.")
                        tf_takes = gr.Slider(1, 2, value=2, step=1, label="📦 Stereo Takes", info="Generate 1 or 2 variations.")
                    with gr.Accordion("🛠️ Zero-Latency UI Mastering Controls", open=False):
                        with gr.Row():
                            tf_master = gr.Checkbox(value=True, label="✨ Auto-Trim Leading Silence & Normalize", info="Zero-latency UI trim.")
                            tf_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Trim Threshold (dB)", info="Silence threshold.")
                            tf_fade = gr.Slider(20.0, 350.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Smooth release.")
                    tf_btn = gr.Button("⚡ Generate 44.1 kHz Stereo SFX (TangoFlux)", variant="primary")

                with gr.Column(scale=2):
                    tf_wav1 = gr.Audio(label="🎧 TangoFlux 44.1 kHz Stereo — Take 1", type="filepath")
                    tf_wav2 = gr.Audio(label="🎧 TangoFlux 44.1 kHz Stereo — Take 2", type="filepath")
                    tf_spec = gr.Image(label="📊 Stereo Waveform & Spectrogram", type="filepath")
                    tf_status = gr.Markdown()

            tf_preset.change(fn=lambda x: x, inputs=[tf_preset], outputs=[tf_prompt])
            tf_btn.click(
                fn=ui_tf_generate,
                inputs=[tf_var, tf_prompt, tf_dur, tf_steps, tf_cfg, tf_seed, tf_takes, tf_master, tf_trim, tf_fade],
                outputs=[tf_wav1, tf_wav2, tf_spec, tf_status],
            )

        with gr.Tab("📈 2. EzAudio-ControlNet Envelope-Cloned SFX"):
            gr.Markdown(
                "### 📈 Rhythm & Loudness Contour Cloning (`EzAudio-ControlNet Energy`)\n"
                "Pick a **Synthetic Classic Windows Rhythm Contour** (e.g. *Ascending Two-Note USB Connect* or *Double Notification Pulse*) "
                "OR upload any `.wav` file — `EzAudio-ControlNet` locks onto its exact energy envelope while synthesizing your new text prompt!"
            )
            with gr.Row():
                with gr.Column():
                    cn_src = gr.Radio(
                        choices=["Built-in Synthetic UI Rhythm Contour", "Upload Custom Reference Audio (.wav / .mp3)"],
                        value="Built-in Synthetic UI Rhythm Contour",
                        label="📐 Envelope Source",
                        info="Use a clean synthetic rhythm contour or your own reference recording.",
                    )
                    cn_synth = gr.Dropdown(
                        choices=list(ENVELOPE_TEMPLATES.keys()),
                        value="Ascending Two-Note Chime — USB Connect (1.1s)",
                        label="🎵 Built-in UI Rhythm Envelope Template",
                        info="Generates a mathematical attack/decay guide curve for ControlNet.",
                    )
                    cn_upload = gr.Audio(label="📂 Or Upload Custom Reference Audio", type="filepath")
                    cn_prompt = gr.Textbox(
                        value="Warm organic wooden marimba and crystal bell chime",
                        lines=2,
                        label="📝 Target Material / Instrument Prompt",
                        info="Describe the sound texture that should follow the reference energy contour.",
                    )
                    with gr.Row():
                        cn_cond_scale = gr.Slider(0.2, 2.0, value=1.0, step=0.1, label="📐 ControlNet Conditioning Scale", info="How strictly the output follows the reference energy envelope (default 1.0).")
                        cn_cfg = gr.Slider(1.0, 8.0, value=3.5, step=0.5, label="🧭 Guidance Scale", info="Classifier-free guidance scale (default 3.5).")
                        cn_steps = gr.Slider(20, 100, value=50, step=5, label="⚡ DDIM Steps", info="Denoising steps (default 50).")
                        cn_seed = gr.Number(value=42, precision=0, label="🎲 Seed", info="Random seed.")
                    with gr.Row():
                        cn_master = gr.Checkbox(value=True, label="✨ Zero-Latency UI Mastering", info="Auto-trim leading silence.")
                        cn_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Trim (dB)", info="Silence threshold.")
                        cn_fade = gr.Slider(20.0, 350.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Smooth tail fade.")
                    cn_btn = gr.Button("📈 Generate Envelope-Locked Sound Effect", variant="primary")
                with gr.Column():
                    cn_out = gr.Audio(label="🎧 ControlNet Generated Output", type="filepath")
                    cn_ref_prev = gr.Audio(label="📐 Reference Envelope Guide Used", type="filepath")
                    cn_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    cn_status = gr.Markdown()

            cn_btn.click(
                fn=ui_ez_controlnet,
                inputs=[cn_src, cn_synth, cn_upload, cn_prompt, cn_cond_scale, cn_cfg, cn_steps, cn_seed, cn_master, cn_trim, cn_fade],
                outputs=[cn_out, cn_ref_prev, cn_spec, cn_status],
            )

        with gr.Tab("🩹 3. EzAudio Seam-Free Audio Inpainting"):
            with gr.Row():
                with gr.Column():
                    ez_mod_dd = gr.Dropdown(
                        choices=["s3_xl (EzAudio-XL — Best Fidelity)", "s3_l (EzAudio-L — Fast)"],
                        value="s3_xl (EzAudio-XL — Best Fidelity)",
                        label="🎛️ EzAudio Checkpoint",
                        info="Select s3_xl or s3_l.",
                    )
                    ez_src = gr.Audio(label="📂 Upload Audio to Edit / Inpaint", type="filepath")
                    ez_prompt = gr.Textbox(value="Bright metallic chime flourish", label="📝 Replacement Prompt", info="What should be synthesized inside the masked window.")
                    with gr.Row():
                        ez_start = gr.Slider(0.0, 10.0, value=1.0, step=0.1, label="⏱️ Mask Start (s)", info="Start of region to replace.")
                        ez_len = gr.Slider(0.2, 8.0, value=1.5, step=0.1, label="📏 Mask Length (s)", info="Duration of region to replace.")
                        ez_bound = gr.Slider(0.5, 4.0, value=2.0, step=0.5, label="🔗 Context Boundary (s)", info="Surrounding audio context preserved.")
                    ez_btn = gr.Button("🩹 Inpaint Audio Segment", variant="primary")
                with gr.Column():
                    ez_out = gr.Audio(label="🎧 Inpainted Audio Output", type="filepath")
                    ez_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    ez_status = gr.Markdown()

            ez_btn.click(
                fn=ui_ez_inpaint,
                inputs=[ez_mod_dd, ez_src, ez_prompt, ez_start, ez_len, ez_bound],
                outputs=[ez_out, ez_spec, ez_status],
            )

        with gr.Tab("🪟 4. Dual-Engine Windows Sound Event Pack Builder"):
            gr.Markdown(
                "### 🪟 Windows 10/11 Sound Scheme Builder — TangoFlux Stereo OR ControlNet Rhythm-Locked!\n"
                "Choose **TangoFlux (44.1 kHz Stereo)** for wide stereo chimes, or **EzAudio-ControlNet (Rhythm-Locked)** "
                "to lock every Windows event (`DeviceConnect`, `DeviceDisconnect`, `WindowsLogon`, `SystemNotification`) to classic multi-note rhythm contours!"
            )
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        tw_engine = gr.Radio(
                            choices=[
                                "TangoFlux CRPO (44.1 kHz Stereo — Fast & Wide)",
                                "EzAudio-ControlNet (Rhythm-Contour Locked)",
                            ],
                            value="TangoFlux CRPO (44.1 kHz Stereo — Fast & Wide)",
                            label="⚙️ Pack Generation Engine",
                            info="Select TangoFlux for stereo flow matching or EzAudio-ControlNet for rhythm-locked contours.",
                        )
                        tw_scheme = gr.Textbox(value="TangoFlux_Stereo_OS", label="🏷️ Windows Sound Scheme Name", info="Display name in mmsys.cpl.")
                    tw_theme = gr.Dropdown(
                        choices=list(SONIC_THEME_PRESETS.keys()),
                        value=list(SONIC_THEME_PRESETS.keys())[0],
                        label="🎨 Sonic Theme Preset",
                        info="Cohesive aesthetic across all 16 system events.",
                    )
                    tw_custom = gr.Textbox(value="", label="✍️ Custom Theme Style Override (Optional)", info="Leave blank to use preset.")
                    tw_events = gr.CheckboxGroup(
                        choices=list(WINDOWS_SYSTEM_EVENTS.keys()),
                        value=DEFAULT_SELECTED_WIN_EVENTS,
                        label="🔔 Windows System Events to Generate",
                        info="Select which Windows events to include.",
                    )
                    with gr.Row():
                        tw_steps = gr.Slider(20, 80, value=45, step=5, label="⚡ TangoFlux Steps", info="Default 45.")
                        tw_cfg = gr.Slider(1.0, 8.0, value=4.5, step=0.5, label="🧭 Guidance Scale", info="Default 4.5.")
                        tw_seed = gr.Number(value=42, precision=0, label="🎲 Base Seed", info="Seed.")
                    with gr.Row():
                        tw_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Zero-Latency Trim (dB)", info="Removes pre-attack dead air.")
                        tw_fade = gr.Slider(40.0, 300.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Click-free release.")
                    tw_btn = gr.Button("🪟 Build Complete Windows Sound Scheme (.zip + .ps1 + .reg)", variant="primary")
                    with gr.Accordion("🎯 Single-Event Re-Roll & Pitch Customizer (Update 1 Event Without Regenerating Pack)", open=False):
                        with gr.Row():
                            tw_rr_ev = gr.Dropdown(choices=list(WINDOWS_SYSTEM_EVENTS.keys()), value="DeviceConnect", label="🔔 Event to Re-Roll / Replace", info="Pick a single Windows event to regenerate.")
                            tw_rr_seed = gr.Number(value=-1, precision=0, label="🎲 New Seed (-1 = Random)", info="Fresh seed.")
                            tw_rr_pitch = gr.Slider(-6.0, 6.0, value=0.0, step=0.5, label="🎵 Pitch Transpose (Semitones)", info="Semitone shift.")
                        tw_rr_prompt = gr.Textbox(value="", placeholder="Optional custom prompt override for this event...", label="✍️ Single-Event Prompt Override (Optional)", info="Leave blank to use theme prompt.")
                        tw_rr_btn = gr.Button("🔄 Re-Roll Single Event & Update Pack")
                with gr.Column(scale=2):
                    tw_zip = gr.File(label="📦 Download Windows Sound Pack (.zip)")
                    tw_prev = gr.Audio(label="🎧 Sample Event Preview", type="filepath")
                    tw_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    tw_status = gr.Markdown()
            tw_board = gr.HTML(label="🎹 Interactive Dual-Engine Windows Sound Scheme Audition Board")

            tw_btn.click(
                fn=ui_tfez_windows_pack,
                inputs=[tw_engine, tw_scheme, tw_theme, tw_custom, tw_events, tw_steps, tw_cfg, tw_seed, tw_trim, tw_fade],
                outputs=[tw_zip, tw_prev, tw_spec, tw_board, tw_status],
            )
            tw_rr_btn.click(
                fn=ui_tfez_reroll_single_event,
                inputs=[tw_engine, tw_scheme, tw_theme, tw_custom, tw_rr_ev, tw_rr_prompt, tw_rr_seed, tw_rr_pitch, tw_steps, tw_cfg, tw_trim, tw_fade],
                outputs=[tw_zip, tw_prev, tw_spec, tw_board, tw_status],
            )

    demo.load(
        fn=lambda: f"🟢 **TangoFlux + EzAudio-ControlNet Studio Ready** on `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}`!",
        inputs=None,
        outputs=[banner],
    )

clear_output()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=False, show_error=True)

In [ ]:
#@title Step 5: Keep-Alive Helper & GPU VRAM Monitor
import time
import torch

KEEP_ALIVE_MINUTES = 0 #@param {type:"integer"}

try:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(0) / (1024 ** 3)
        resv = torch.cuda.memory_reserved(0) / (1024 ** 3)
        print(f"📊 GPU VRAM Status — Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB")
    if KEEP_ALIVE_MINUTES > 0:
        print(f"⏳ Keeping Gradio session alive for {KEEP_ALIVE_MINUTES} minutes...")
        for m in range(KEEP_ALIVE_MINUTES):
            time.sleep(60)
            print(f"  ✓ Active: {m + 1}/{KEEP_ALIVE_MINUTES} min elapsed")
    else:
        print("ℹ️ Keep-alive loop skipped (KEEP_ALIVE_MINUTES=0). Set > 0 if running unattended.")
except KeyboardInterrupt:
    print("⏹️ Keep-alive stopped by user.")
except Exception as e:
    print(f"Keep-alive warning: {e}")

In [ ]:
#@title Step 6: Quick Test — TangoFlux 44.1 kHz Stereo SFX Synthesis
import os
from IPython.display import Audio, FileLink, display

QUICKTEST_PROMPT = "Two-note ascending crystalline glass marimba notification chime with soft reverb" #@param {type:"string"}
QUICKTEST_DURATION = 2.5 #@param {type:"number"}
QUICKTEST_SEED = 42 #@param {type:"integer"}

try:
    wav_path, _, _, msg = ui_tf_generate(
        variant="declare-lab/TangoFlux",
        prompt=QUICKTEST_PROMPT,
        duration=QUICKTEST_DURATION,
        steps=40,
        guidance_scale=4.5,
        seed=QUICKTEST_SEED,
        num_takes=1,
        apply_mastering=True,
        trim_db=-42.0,
        fade_out_ms=120.0,
    )
    print(msg)
    if wav_path and os.path.exists(wav_path):
        display(Audio(filename=wav_path))
        display(FileLink(wav_path))
except Exception as e:
    print(f"Quick test error: {e}")

In [ ]:
#@title Step 7: Headless Batch Windows Sound Event Pack Export (TangoFlux Stereo)
import os
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_SCHEME_NAME = "TangoFlux_Stereo_Pack" #@param {type:"string"}
BATCH_THEME = "Frutiger Aero Crystal Glass" #@param ["Frutiger Aero Crystal Glass", "Cyberpunk Neon HUD", "Cozy Wooden Kalimba & Mallet", "Sci-Fi Spaceship Bridge", "Retro 16-Bit Chiptune Arcade", "Warm Lo-Fi Analog Tape Synth", "Minimalist Tactile Micro-Foley", "Dark Fantasy Crystal & Harp"]
BATCH_SEED = 42 #@param {type:"integer"}

try:
    event_map = {}
    pack_dir = os.path.join(OUTPUT_DIR, f"batch_tf_winpack_{BATCH_SEED}")
    os.makedirs(pack_dir, exist_ok=True)
    for idx, ev_key in enumerate(tqdm(DEFAULT_SELECTED_WIN_EVENTS, desc="TangoFlux Stereo Windows Pack")):
        try:
            ev_meta = WINDOWS_SYSTEM_EVENTS[ev_key]
            dur = float(ev_meta["duration"])
            prompt = build_windows_event_prompt(ev_key, BATCH_THEME, "")
            arr, sr, _, _ = TFEZ_STUDIO.generate_tangoflux(
                variant="declare-lab/TangoFlux",
                prompt=prompt,
                duration=max(dur + 0.4, 1.5),
                steps=40,
                guidance_scale=4.5,
                seed=BATCH_SEED + idx,
            )
            mastered = master_ui_sound(arr, sr, target_duration_s=dur)
            fpath = os.path.join(pack_dir, f"{ev_key}.wav")
            save_pcm16_wav(fpath, mastered, sr)
            event_map[ev_key] = {"path": fpath, "filename": f"{ev_key}.wav", "prompt": prompt}
        except Exception as item_e:
            print(f"⚠️ Event {ev_key} skipped: {item_e}")

    if event_map:
        zip_out = package_windows_sound_scheme_zip(BATCH_SCHEME_NAME, event_map, BATCH_THEME, "TangoFlux CRPO (44.1 kHz Stereo)", OUTPUT_DIR)
        print(f"✅ Packaged Windows Sound Scheme: {zip_out}")
        display(FileLink(zip_out))
except Exception as e:
    print(f"Batch Windows Pack error: {e}")